# langChain Model 基础学习
我们刚刚完成了langchain前期必要的环境配置和依赖安装 现在开始学习langchain的基础使用

## 引入env环境
引入环境变量记录的deepseek APIkey

In [ ]:
import { load } from "dotenv";
await load({ 
export: true,
envPath:'../.env'
});

{ OPENAI_API_KEY: "sk-be17cc28883b4b76b8924f23e4ca522b" }

# LCEL
LCEL 是一种 把 LangChain 的各个组件“拼”在一起的声明式语法 。

可以把构建 AI 应用想象成一条 流水线（Pipeline） ：

1. 提示词模板（Prompt） ：负责把用户的输入包装成大模型听得懂的话。
2. 大模型（LLM） ：负责思考并生成回复。
3. 输出解析器（Output Parser） ：负责把大模型乱七八糟的输出整理成你想要的格式（比如纯文本、JSON 等）。

在没有 LCEL 之前，你需要写很多啰嗦的代码来把这三步连起来：先调用 Prompt 获取字符串，再把字符串传给 LLM 获取对象，再把对象传给 Parser。要写很多啰嗦的代码来把这三步连起来：先调用 Prompt 获取字符串，再把字符串传给 LLM 获取对象，再把对象传给 Parser。有了 LCEL，你只需要用管道符（在 JS/TS 里是 .pipe() 方法）把它们连起来就行了！

```ts
import { ChatOpenAI } from "@langchain/openai";
import { PromptTemplate } from "@langchain/core/prompts";
import { StringOutputParser } from "@langchain/core/output_parsers";

// 1. 准备组件
const prompt = PromptTemplate.fromTemplate("给我讲一个关于 {topic} 的冷笑话");
const model = new ChatOpenAI({ modelName: "gpt-3.5-turbo" });
const parser = new StringOutputParser();

// 2. 🌟 这就是 LCEL 的灵魂：用 .pipe() 把它们串起来变成一条链 (Chain)
const chain = prompt.pipe(model).pipe(parser);

// 3. 执行这条链
const result = await chain.invoke({ topic: "程序员" });
console.log(result); 
// 输出: "为什么程序员总分不清万圣节和圣诞节？因为 Oct 31 == Dec 25。"
```
在这个实例中只需要调用chain.invoke()方法即可 数据机会自动通过流水线传递最后返回结果

LCEL 从底层设计的目标就是支持 从原型到生产 完整流程不需要修改任何代码，也就是我们在写的任何原型代码不需要太多的改变就能支持生产级别的各种特性（比如并行、steaming 等），具体来说会有这些优势：

- 并行，只要是整个 chain 中有可以并行的步骤就会自动的并行，来减少使用时的延迟。
- 自动的重试和 fallback。大部分 chain 的组成部分都有自动的重试（比如因为网络原因的失败）和回退机制，来解决很多请求的出错问题。 而不需要我们去写代码 cover 这些问题。
- 对 chain 中间结果的访问，在旧的写法中很难访问中间的结果，而 LCEL 中可以方便的通过访问中间结果来进行调试和记录。
LCEL 会自动支持 LangSimith 进行可视化和记录。这是 langchain 官方推出的记录工具，可以记录一条 chian 运行过程中的大部分信息，来方便调试 LLM 找到是哪些中间环节的导致了最终结果较差。这部分我们会在后续的章节中涉及到。

一条 Chain 组成的每个模块都是继承自 Runnable 这个接口，而一条 Chain 也是继承自这个接口，所以一条 Chain 也可以很自然的成为另一个 Chain 的一个模块。并且所有 Runnable 都有相同的调用方式。 所以在我们写 Chain 的时候就可以自由组合多个 Runnable 的模块来形成复杂的 Chain。

对于任意 Runnable 对象，其都会有这几个常用的标准的调用接口：

- invoke 基础的调用，并传入参数
- batch 批量调用，输入一组参数
- stream 调用，并以 stream 流的方式返回数据
- streamLog 除了像 stream 流一样返回数据，并会返回中间的运行结果
Talk is cheap，让我们来看 code 演示，其中会涉及到很多 Langchain 中陌生的概念，大家可以简单从它的表现中理解，我们会在后续的章节中深入介绍。

## 重点掌握的几个接口

- invoke 基础的调用，并传入参数
- batch 批量调用，输入一组参数
- stream 调用，并以 stream 流的方式返回数据
- streamLog 除了像 stream 流一样返回数据，并会返回中间的运行结果

## 重要名词

Runnable 可运行对象 是核心标准接口 无论是提示词还是大模型 都需要实现这个接口 Runnable就是LCEL的基本单位

只要一个对象实现了 Runnable 接口，它就一定具备以下这几个极其重要的方法：

- .invoke(input) ： 基础调用 。单次传入一个输入，等待它执行完毕，返回一个输出。
- .stream(input) ： 流式输出 。调用后不会等待全部完成，而是一点点（chunk）返回结果，常用于前端打字机效果。
- .batch(inputs) ： 批量处理 。传入一个数组，它会自动并发处理这些输入。
- .pipe(nextRunnable) ： 核心串联魔法 。将当前的 Runnable 和下一个 Runnable 连接起来，生成一个新的、更大的 Runnable （叫做 RunnableSequence ）。


# invoke

首先我们使用最基础的ChatOpenAI,这是一个`Runnable`对象, 其中 HumanMessage 你可以理解成构建一个用户输入，各种 Message 的介绍我们会在后续章节中展开介绍。 注意这里 chatModel 需要的输入是一个 Message 的列表。


```ts
// 使用ChatOpenAi模型调用deepseek
import { ChatOpenAI } from "@langchain/openai";
import { HumanMessage } from "@langchain/core/messages";

const model = new ChatOpenAI({
    modelName: "deepseek-chat",
    configuration: {
        baseURL: "https://api.deepseek.com",
    }
});

await model.invoke([
    new HumanMessage("Tell me a joke")
])
```

为了方便结果的展示,我们会加入一个简单的`StringOutputParser`处理输出,可以简单理解为将模型返回的复杂对象提取出最核心的字符串,更详细的'OutputParser'后续会解释

解释一下下面的代码:
1. .pipe() 是什么方法？
.pipe() （中文常译为“管道”）是所有 Runnable 对象自带的一个拼接方法。
它的工作原理就像现实生活中的 自来水管拼接 ，或者 Linux 系统里的管道符（ | ）：
- 它负责把 左边组件的“输出” ，无缝地作为 右边组件的“输入” 。
- 在你这行代码中， model （大模型）的输出是一个复杂的包含元数据的 AIMessage 对象， .pipe() 会自动把这个对象塞给右边的 outputParser （输出解析器）
2. 为什么要在这里创建一个 simpleChain？
创建 simpleChain 的核心目的是 把多个繁琐的步骤打包成一个统一的流水线（Chain） ，从而简化代码调用。
创建 simpleChain 的三大好处:
   - 隐藏复杂的中间状态 ：你不需要再去关心大模型返回的具体是 AIMessage 还是什么复杂的 JSON 结构， simpleChain 内部自动消化了这些转换，你直接输入字符串，拿到的就是解析好的纯文本结果。
   - 代码高度复用 ： simpleChain 作为一个定义好的业务流，可以在项目的任何地方反复调用，而不需要每次都写“调用模型 -> 提取文本”的重复代码。
   - 完美支持流式输出 (Streaming) ：这也是最强大的一点。如果你之后想做类似 ChatGPT 那样的打字机效果（一个字一个字往外蹦），只要流水线是用 .pipe() 连起来的，你直接调用 simpleChain.stream() 就能自动实现，极其方便。。


In [7]:
// 使用ChatOpenAi模型调用deepseek
import { ChatOpenAI } from "@langchain/openai";
import { HumanMessage } from "@langchain/core/messages";
import {StringOutputParser} from "@langchain/core/output_parsers";

const model = new ChatOpenAI({
    modelName: "deepseek-chat",
    configuration: {
        baseURL: "https://api.deepseek.com",
    }
});
// 创建一个字符串输出解析器
const outputParser = new StringOutputParser();

const simpleChain = model.pipe(outputParser);

await simpleChain.invoke([
    new HumanMessage("讲个笑话")
])


"一个程序员去面试。\n" +
  "\n" +
  "面试官问：“你懂什么编程语言？”\n" +
  "\n" +
  "程序员答：“Java、Python、C++、JavaScript……”\n" +
  "\n" +
  "面试官眼睛一亮：“很好！那你先写个程序，输出‘Hello World’吧。”\n" +
  "\n" +
  "程序员沉思片刻，然后认真地问：“请问您要哪种语言的？需要适配什么操作系统？考虑跨平台兼容吗？需要支持多线程高并发吗？另外，UI要什么风格？后端要微服务架构吗？……”\n" +
  "\n" +
  "面试官沉默三秒，说：“我只要打印到纸上。”\n" +
  "\n" +
  "程序员：“哦，那得先选打印机驱动。”"

在 LCEL 中，使用` .pipe()` 方法来组装多个` Runnable` 对象形成完整的` Chain，`可以看到我们是用对单个模块同样的 `invoke `方法去调用整个` chain`。 因为无论是单个模块还是由模块组装而成的多个 `chain `都是` Runnable。`

# batch
然后我们尝试对这个基础的 Chain 进行批量调用，用起来也非常简单, 使用 `batch` 方法即可,`invoke`方法是单次调用,`batch`方法是批量调用

In [ ]:
await simpleChain.batch([
  [new HumanMessage("你是谁")],
  [new HumanMessage("你好吗")],
])

可见,使用 `batch` 方法可以批量调用 chain, 且每个调用的输入是一个数组, 输出也是一个数组

# stream
因为大模型的很多调用是一段一段返回的,如果等到完整内容返回给用户,就会让用户长时间的等待,非常影响用户体验,所以LCEL就支持steaming方法,我们依旧可以使用我们定义的基础Chain来调用steam流式输出

In [ ]:
const stream = await simpleChain.stream([
  new HumanMessage("Tell me a joke")
])

for await (const chunk of stream){
  console.log(chunk)
}

可见其返回值是一块一块返回的,用户能够立即开始接收返回值,而不需要等待完整内容返回

# fallback

fallback(备胎) 是一种容错机制,可以吧fallback作为备用系统, 当主系统失败的时候(比如LLM的API失败 网络超时等情况) LangChain会自动切换到你指定的备用方案,而不是直接崩溃

`withFallbacks`是任何runnable都有的函数,可以给当前runnable对象添加fallback 然后生成带一个fallback的`runnableWithFallback`对象, 这适合我们将自己的fallback逻辑增加到lcel中



In [20]:
// 我们创建一个一定会失败的 llm ：
import { ChatOpenAI } from "@langchain/openai";

// 这个模型是注定不能被调用的
const fakeLLM = new ChatOpenAI({
    openAIApiKey: "fake-key-123",
    maxRetries: 0,
});

// await fakeLLM.invoke("你好")

因为大多 `runnable` 都自带出错重试的机制，所以我们在这将重试的次数 `maxRetries` 设置为 0。

然后，我们创建一个可以成功的 llm，并设置为 fallback：

In [21]:
// 为了方便后续使用 我们将deepseek模型的创建方式封装为一个工厂函数
function getDeepSeekBot(){
  return new ChatOpenAI({
    modelName: "deepseek-chat",
    configuration: {
        baseURL: "https://api.deepseek.com",
    }
  })
}

const realLLm = getDeepSeekBot()

const llmWithFallbacks = fakeLLM.withFallbacks({
  fallbacks: [realLLm]
})

await llmWithFallbacks.invoke("你好") 


AIMessage {
  lc_serializable: true,
  lc_kwargs: {
    content: "你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意为你提供支持！\n" +
      "\n" +
      "我可以帮你解答各种问题，协助处理文档，进行创作和分析等等。有什么我可以为你做的吗？",
    additional_kwargs: { function_call: undefined, tool_calls: undefined },
    response_metadata: {}
  },
  lc_namespace: [ "langchain_core", "messages" ],
  content: "你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意为你提供支持！\n" +
    "\n" +
    "我可以帮你解答各种问题，协助处理文档，进行创作和分析等等。有什么我可以为你做的吗？",
  name: undefined,
  additional_kwargs: { function_call: undefined, tool_calls: undefined },
  response_metadata: {
    tokenUsage: { completionTokens: 64, promptTokens: 5, totalTokens: 69 },
    finish_reason: "stop"
  }
}